In [1]:
import torch

In [2]:
from sentence_transformers import SentenceTransformer

In [3]:
model = SentenceTransformer('bert-base-nli-mean-tokens')

  0%|          | 0.00/405M [00:00<?, ?B/s]

Some weights of the model checkpoint at /Users/kazotogbah/.cache/torch/sentence_transformers/sbert.net_models_bert-base-nli-mean-tokens/0_BERT were not used when initializing BertModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [5]:
import pandas as pd
path = "/Users/kazotogbah/Desktop/ResearchDesignForBusinessAnalytics/HMH_box/OneCMS_Learning_Spine_English_Language_Arts2021012117.xlsx" 
raw_skills = pd.io.excel.read_excel(path)

In [6]:
skills = raw_skills[5:]

In [7]:
skills_clean=skills.rename(columns={"Learning Spine Title:": "Level 1: Domain", "Learning Spine: English Language Arts": "Level 2: Strand",
                          "Unnamed: 2":"Level 3: Substrand 1","Unnamed: 3":"Level 4: Substrand 2","Unnamed: 4":"Level 5: Substrand 3",
                          "Unnamed: 5":"Skill Title","Unnamed: 6":"Skill Description","Unnamed: 7":"Skill GUID","Unnamed: 8":"Skill Code",
                          "Unnamed: 9":"Lower Grade","Unnamed: 10":"Upper Grade"})

In [8]:
skills_clean=skills_clean.reset_index(drop=True)

In [9]:
skills_clean['Skill Description'][0]

"Identify an informational text's intended audience while reading, and analyze how it affects the author's development of a text"

In [10]:
type(skills_clean['Skill Description'])

pandas.core.series.Series

In [11]:
sentence_embeddings = model.encode(skills_clean['Skill Description'])

In [12]:
sentence_embeddings.shape

(2294, 768)

---
#### Now what we do is take those embeddings and find the cosine similarity between each

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
cosine_similarity(
    [sentence_embeddings[0]],
    sentence_embeddings[1:]
)

array([[0.87060356, 0.89145947, 0.8928533 , ..., 0.5376834 , 0.69654274,
        0.45685524]], dtype=float32)

---
# Involved — Transformers And PyTorch

In [15]:
from transformers import AutoTokenizer, AutoModel
import torch

In [16]:
model_name='sentence-transformers/bert-base-nli-mean-tokens'

In [17]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

Downloading:   0%|          | 0.00/461 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/232k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/112 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at sentence-transformers/bert-base-nli-mean-tokens were not used when initializing BertModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [18]:
# initialize dictionary to store tokenized sentences
tokens = {'input_ids': [], 'attention_mask': []}

In [19]:
for sentence in skills_clean['Skill Description']:
    new_tokens=tokenizer.encode_plus(sentence,max_length=128,truncation=True,
                          padding='max_length',return_tensors='pt')
    tokens['input_ids'].append(new_tokens['input_ids'][0])
    tokens['attention_mask'].append(new_tokens['attention_mask'][0])


In [20]:
tokens['input_ids']

[tensor([  101,  6709,  2019,  2592,  2389,  3793,  1005,  1055,  3832,  4378,
          2096,  3752,  1010,  1998, 17908,  2129,  2009, 13531,  1996,  3166,
          1005,  1055,  2458,  1997,  1037,  3793,   102,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

In [21]:
type(tokens['input_ids'])

list

In [23]:
tokens['input_ids'][0]

tensor([  101,  6709,  2019,  2592,  2389,  3793,  1005,  1055,  3832,  4378,
         2096,  3752,  1010,  1998, 17908,  2129,  2009, 13531,  1996,  3166,
         1005,  1055,  2458,  1997,  1037,  3793,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0])

In [22]:
type(tokens['input_ids'][0])

torch.Tensor

In [24]:
tokens['input_ids']=torch.stack(tokens['input_ids'])

In [25]:
tokens['attention_mask']=torch.stack(tokens['attention_mask'])

In [26]:
tokens['input_ids']

tensor([[  101,  6709,  2019,  ...,     0,     0,     0],
        [  101,  3305,  2008,  ...,     0,     0,     0],
        [  101,  2096,  3752,  ...,     0,     0,     0],
        ...,
        [  101,  4339, 24040,  ...,     0,     0,     0],
        [  101,  4339,  2365,  ...,     0,     0,     0],
        [  101,  4339,  5878,  ...,     0,     0,     0]])

In [28]:
tokens['input_ids'].shape

torch.Size([2294, 128])

In [27]:
tokens['attention_mask']

tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])

In [1]:
outputs = model(**tokens)

NameError: name 'model' is not defined

In [ ]:
outputs

In [ ]:
outputs.keys()

In [ ]:
embeddings=outputs.last_hidden_state

In [ ]:
embeddings

In [ ]:
embeddings.shape

In [ ]:
attention=tokens['attention_mask']

In [ ]:
attention.shape

In [ ]:
attention.unsqueeze(-1).shape

In [ ]:
mask=attention.unqueeze(-1).expand(embedding.shape).float

In [ ]:
mask_embeddings = embeddings * mask

In [ ]:
mask_embeddings.shape

In [ ]:
summed = torch.sum(mask_embeddings,1)
summed.shape

In [ ]:
counts = torch.clamp(mask.sum(1),min= 1e-9)
counts.shape

In [ ]:
mean_pooled = summed/counts
mean_pooled.shape

In [ ]:
mean_pooled

---
# cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# convert from PyTorch tensor to numpy array
mean_pooled = mean_pooled.detach().numpy()

In [ ]:
# calculate
cosine_similarity(
    [mean_pooled[0]],
    mean_pooled[1:]
)